# einops-einsum — ex5: attention scores QK^T (batched + reduce + matmul-like)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-einsum`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five `einops.einsum` patterns that ramp from elementwise product → matrix multiply → omit-to-reduce → batched matmul → attention QK^T. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import einsum

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Deep Learning` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `einops-einsum`**, which bridges to the bank subtopic `Einops: Deep Learning` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-einsum"
DD_SUBTOPIC = "Einops: Deep Learning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## einops.einsum — quick refresher

`einsum(*tensors, pattern)` performs sum-contraction over named indices:
1. **Elementwise** — `'i j, i j -> i j'` multiplies pointwise (no reduction).
2. **Matmul** — `'i k, k j -> i j'` contracts the shared `k` (sum-reduce).
3. **Single-operand reduce** — `'i j -> i'` sums over `j` (no second tensor needed).
4. **Batched** — `'b i k, b k j -> b i j'` carries `b` through, contracts `k`.

**The two rules:**
- An index that appears on input AND output → preserved (broadcast-like).
- An index that appears on input but NOT on output → sum-contracted.

### Exercise 5 — attention scores QK^T (batched + reduce + matmul-like)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Synthesize batched einsum with index-contraction to produce attention scores (QK^T) without reshaping.
> Keywords: attention, qkt, integration, multi-kc
> ```

**KCs targeted:** `einsum-matmul-contraction`, `einsum-batched`, `einsum-attention-scores`

Implement `ex5_attention_scores(q, k)` to compute pre-softmax attention scores.

Input shapes: `q` is `(b, q_len, d)`, `k` is `(b, k_len, d)`. Output shape: `(b, q_len, k_len)`. Each `out[b, i, j]` is the inner product `sum_d q[b, i, d] * k[b, j, d]`.

This is QK^T from a Transformer attention head, batched over `b`. Notice **you do not transpose** `k` — einsum handles the index alignment for you. `d` appears on both inputs and not on the output, so it's contracted.

> ⚠️ **Integrative exercise.** This combines 3+ KCs in one pattern; empirical work (Lohr et al. ITiCSE 2025) shows 3-concept LLM-generated exercises drop from ~94% to ~40% solvability. Expect a step in difficulty here vs Exercises 1-4.

In [ ]:
def ex5_attention_scores(q: Tensor, k: Tensor) -> Tensor:
    """Attention scores. (b, q_len, d) @ (b, k_len, d)^T → (b, q_len, k_len).

    Each out[b, i, j] = sum_d q[b, i, d] * k[b, j, d].
    """
    raise NotImplementedError()


def _test_ex5():
    b, q_len, k_len, d = 2, 3, 5, 4
    q = t.randn(b, q_len, d)
    k = t.randn(b, k_len, d)
    out = ex5_attention_scores(q, k)
    assert out.shape == (b, q_len, k_len), f'expected ({b},{q_len},{k_len}), got {out.shape}'
    # Ground truth via explicit batched matmul with manual transpose.
    expected = q @ k.transpose(-2, -1)
    assert t.allclose(out, expected, atol=1e-5), 'values differ from q @ k.transpose(-2,-1)'
    _dd_passed.add('ex5')
    print("ex5 ✓")

_test_ex5()

<details><summary>Solution</summary>

```python
def ex5_attention_scores(q: Tensor, k: Tensor) -> Tensor:
    return einsum(q, k, 'b q d, b k d -> b q k')
```

**Reading the pattern.**
- `b` appears in both inputs and on the output → carried through (batch).
- `q` only appears in the first input and on the output → preserved.
- `k` only appears in the second input and on the output → preserved.
- `d` appears in both inputs but **not** on the output → contracted (this is the dot product).

**Why this is the integrative case.** You're simultaneously batching (KC #4), preserving two independent non-contracted axes from different operands (extends KC #2 from `i,k → k,j` to a non-square layout), and letting omitting `d` do the reduction (KC #3). No transpose, no rearrange, no reshape — the pattern string carries the full intent.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex5'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex5',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()